In [ ]:
import os
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

In [ ]:
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./postproc/testing/"
os.makedirs(path_to_figs, exist_ok=True)


In [ ]:
path_to_model = "./saving/lin_me/SavedModel/"
path_to_data = "./data/test.hdf5"
path_to_norm = "./data/norm/"

In [ ]:
model = tf.keras.models.load_model(path_to_model)

In [ ]:
norm = {}
for identifier in ("node_attr", "edge_attr", "target"):
    norm[identifier] = {}
    for qoi in ("mean", "std"):
        filename = path_to_norm + f"{identifier}_{qoi}.txt"
        norm[identifier][qoi] = np.loadtxt(filename).reshape(1,1,-1)

def denormalize(x, identifier, active=True):
    if active:
        x = x * norm[identifier]["std"] + norm[identifier]["mean"]
        return np.exp(x.squeeze()) - 1e-13
    else:
        return x

In [ ]:
def get_data(data, denorm=True):
    # Collect data
    x = [data[k] for k in ("node_attr", "edge_attr")]
    y = data["target"]
    # Make prediction
    y_pred = model(x).numpy()
    # Denormalize data
    t = denormalize(x[1], "edge_attr")[:,-1,-1]
    y = denormalize(y, "target", denorm)
    y_pred = denormalize(y_pred, "target", denorm)
    return t, y, y_pred

In [ ]:
def plot_data(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.close()

In [ ]:
data = h5todict(path_to_data)
for (k, datai) in data.items():
    t, y, y_pred = get_data(datai, denorm=True)
    plot_data(
        x=t[1:],
        y=y_pred[1:],
        xref=t[1:],
        yref=y[1:],
        labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
        scales=['log', 'log'],
        names=[r'O$_2^{(%d)}$' % (i+1) for i in range(y.shape[1])],
        figname=path_to_figs + f"/{k}"
    )